# Solutions: Debugging Strategies

These are the reference solutions for the exercises in `02_debugging_strategies.ipynb`.
Try to solve the exercises yourself first before checking here!

In [ ]:
import sys
sys.path.insert(0, "../../../")
from src.checks import check_equal, check_type, check_contains

print("Setup complete.")

---
## Exercise 1 Solution: Debug the Averaging Function

**The bug:** `total / (len(numbers) - 1)` divides by one less than the list length.
For a 3-element list it divides by 2 instead of 3.

**How to find it:** Add print statements to see the intermediate values.

In [ ]:
# Step 1: Use print debugging to trace the issue
def compute_average_debug(numbers):
    total = 0
    for n in numbers:
        total += n
        print(f"  [DEBUG] After adding {n}: total = {total}")
    
    divisor = len(numbers) - 1  # This is the bug
    print(f"  [DEBUG] Divisor (len - 1): {divisor}")
    print(f"  [DEBUG] Expected divisor (len): {len(numbers)}")
    
    result = total / divisor
    print(f"  [DEBUG] Result: {result}")
    return result

print("Tracing compute_average_debug([2, 4, 6]):")
compute_average_debug([2, 4, 6])
# Output makes it clear: total=12, divisor=2, result=6.0 — but expected 4.0
# The divisor should be 3 (len), not 2 (len - 1)

In [ ]:
# Fixed version
def compute_average_fixed(numbers):
    """Compute the average of a list of numbers."""
    assert isinstance(numbers, list), "numbers must be a list"
    assert len(numbers) > 0, "Cannot average an empty list"
    
    total = 0
    for n in numbers:
        total += n
    return total / len(numbers)  # Fixed: divide by len, not len-1

# Or more Pythonically:
def compute_average_pythonic(numbers):
    """Compute the average using built-in sum()."""
    assert len(numbers) > 0, "Cannot average an empty list"
    return sum(numbers) / len(numbers)

print(f"compute_average_fixed([2, 4, 6]) = {compute_average_fixed([2, 4, 6])}")  # 4.0
print(f"compute_average_fixed([10, 20, 30, 40]) = {compute_average_fixed([10, 20, 30, 40])}")  # 25.0
print(f"Pythonic version: {compute_average_pythonic([2, 4, 6])}")  # 4.0

In [ ]:
check_equal(compute_average_fixed([2, 4, 6]), 4.0, "Average of [2,4,6] should be 4.0")
check_equal(compute_average_fixed([10, 20, 30, 40]), 25.0, "Average of [10,20,30,40] should be 25.0")

---
## Exercise 2 Solution: Add Assert Statements

A complete set of assertions for `compute_weighted_average`.
Each assertion documents an assumption and catches a specific class of bad input.

In [ ]:
def compute_weighted_average_safe(scores, weights):
    """Compute a weighted average. Weights should sum to 1.0."""
    
    # Input type checks
    assert isinstance(scores, list), f"scores must be a list, got {type(scores)}"
    assert isinstance(weights, list), f"weights must be a list, got {type(weights)}"
    
    # Non-empty
    assert len(scores) > 0, "scores cannot be empty"
    assert len(weights) > 0, "weights cannot be empty"
    
    # Matching lengths
    assert len(scores) == len(weights), (
        f"scores and weights must have the same length: "
        f"len(scores)={len(scores)}, len(weights)={len(weights)}"
    )
    
    # Weight validity
    assert all(0 <= w <= 1 for w in weights), f"All weights must be between 0 and 1: {weights}"
    
    # Weights sum to ~1.0 (allow floating point tolerance)
    weight_sum = sum(weights)
    assert abs(weight_sum - 1.0) < 1e-6, f"Weights must sum to 1.0, got {weight_sum:.6f}"
    
    result = sum(s * w for s, w in zip(scores, weights))
    
    # Postcondition: result should be within the range of scores
    assert min(scores) <= result <= max(scores), f"Result {result} is outside score range"
    
    return result

# Valid inputs
result = compute_weighted_average_safe([0.8, 0.6, 0.9], [0.5, 0.3, 0.2])
print(f"Weighted average: {result:.4f}")  # 0.76

# Demonstrate catching bad input
bad_caught = False
try:
    compute_weighted_average_safe([0.8, 0.6], [0.5, 0.3, 0.2])  # Length mismatch
except AssertionError as e:
    bad_caught = True
    print(f"Caught length mismatch: {e}")

# Demonstrate catching bad weights
try:
    compute_weighted_average_safe([0.8, 0.6], [0.5, 0.4])  # Weights sum to 0.9, not 1.0
except AssertionError as e:
    print(f"Caught bad weights: {e}")

In [ ]:
check_equal(round(compute_weighted_average_safe([0.8, 0.6, 0.9], [0.5, 0.3, 0.2]), 4),
            0.76, "Weighted average should be 0.76")
check_equal(bad_caught, True, "Should catch length mismatch with AssertionError")

---
## Exercise 3 Solution: Fix the Silent Logic Error

**The bug:** `if r["score"] < threshold` should be `> threshold`.

**How to find it:** Test with a small known example and compare expected vs. actual output.

In [ ]:
# Debugging approach: print what the condition evaluates to for each record
def filter_high_scores_debug(results, threshold=0.7):
    high = []
    for r in results:
        condition = r["score"] < threshold  # The buggy condition
        print(f"  [DEBUG] id={r['id']}, score={r['score']}, "
              f"(score < threshold) = {condition} "
              f"-- {'INCLUDED' if condition else 'EXCLUDED'}")
        if condition:
            high.append(r)
    return high

test_results = [
    {"id": 1, "score": 0.9},
    {"id": 2, "score": 0.5},
    {"id": 3, "score": 0.8},
    {"id": 4, "score": 0.3},
]

print("Debugging filter_high_scores with threshold=0.7:")
print("(We want scores > 0.7, so IDs 1 and 3 should be included)")
print()
output = filter_high_scores_debug(test_results, threshold=0.7)
print()
print(f"Result: {[r['id'] for r in output]}")
print("Problem clear: the condition is inverted — it's including LOW scores, not high ones!")

In [ ]:
# Fixed version
def filter_high_scores_fixed(results, threshold=0.7):
    """Return only results where score > threshold."""
    high = []
    for r in results:
        if r["score"] > threshold:  # Fixed: > not <
            high.append(r)
    return high

# Or as a one-liner comprehension:
def filter_high_scores_pythonic(results, threshold=0.7):
    return [r for r in results if r["score"] > threshold]

test_results = [
    {"id": 1, "score": 0.9},
    {"id": 2, "score": 0.5},
    {"id": 3, "score": 0.8},
    {"id": 4, "score": 0.3},
]

output = filter_high_scores_fixed(test_results, threshold=0.7)
result_ids = [r["id"] for r in output]
print(f"Fixed result IDs: {result_ids}")  # Should be [1, 3]

In [ ]:
check_equal(result_ids, [1, 3], "Should return IDs 1 and 3 (scores above 0.7)")

---
## Key Patterns from These Solutions

1. **Print the intermediate values, not just the final result.** In Exercise 1, printing `total` and `divisor` immediately revealed the bug.

2. **Test with small examples where you know the expected answer.** `[2, 4, 6]` with expected average `4.0` makes the bug obvious.

3. **Print what the condition evaluates to for each item.** In Exercise 3, printing `(score < threshold) = True/False` made the inversion obvious.

4. **Assertions serve dual purpose:** they catch bugs AND document your intent. When you read `assert abs(weight_sum - 1.0) < 1e-6`, you immediately understand the contract.

5. **Know Python's gotchas for JavaScript developers:**
   - `.sort()` returns `None`, not the list
   - Python doesn't do implicit type coercion in `+` operations
   - `=` inside a condition is a `SyntaxError` (unlike JavaScript where it's silently wrong)